# Entrenamiento de un transformer encoder para atribución de autoría

Dada las limitaciones de hardware de mi equipo local, este notebook está preparado para ejecutarse en Google Colab y usar la GPU que ofrece. El objetivo es entrenar un modelo de transformer encoder para la tarea de atribución de autoría utilizando el corpus `normalized`.

Pasos:

1. Subir las carpetas de cada autor del `corpus/normalized` a una carpeta `corpus` en la ruta `/content/drive/MyDrive/corpus`.
2. Ajustar la ruta en la celda de configuración si es necesario.
3. Añadir el `HF_TOKEN` a los `Secrets` de Colab si no se tiene.
4. Seleccionar GPU como acelerador de hardware en Colab.
5. Ejecutar todas las celdas.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install transformers datasets scikit-learn numpy pandas torchinfo

In [3]:
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict, Counter
from collections.abc import Callable
from dataclasses import dataclass
from typing import Any, Sequence, Mapping
import shutil
import glob

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchinfo import summary
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
)

## Configuración

In [4]:
DATA_DIR = "/content/drive/MyDrive/corpus"
MODEL_NAME = "roberta-base"
OUTPUT_DIR = "/content/drive/MyDrive/outputs/authorship_transformer"

# Limpiar checkpoints antiguos
for path in glob.glob(OUTPUT_DIR + "/checkpoint-*"):
    shutil.rmtree(path)

MAX_LENGTH = 256
STRIDE = 64
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2  # Parar si macro_f1 no mejora durante 2 evaluaciones seguidas
EARLY_STOPPING_THRESHOLD = 0.0  # Mejora mínima exigida en macro_f1
BATCH_SIZE = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42
FP16 = torch.cuda.is_available()
GRADIENT_ACCUMULATION_STEPS = 1

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("CUDA disponible:", torch.cuda.is_available())
print("Usando fp16:", FP16)

CUDA disponible: True
Usando fp16: True


## Funciones auxiliares

Dado que este notebook se ejecutará en Colab, es necesario definir algunas funciones auxiliares para cargar los datos y preparar el dataset para el entrenamiento. Estas funciones se han adaptado para funcionar con la estructura de archivos en Colab.

A diferencia de los notebooks anteriores, donde las utilidades estaban organizadas en módulos separados, aquí se incluyen directamente en el notebook para facilitar su ejecución sin necesidad de importar desde otros archivos.

In [5]:
# @title Funciones auxiliares
def is_valid_text_file(path: Path) -> bool:
    """Check whether a path points to a visible `.txt` file.

    Args:
        path: Path to validate.

    Returns:
        True if the path exists as a file, has a `.txt` suffix case-insensitively,
        and its file name does not start with a dot; otherwise False.
    """
    return path.is_file() and path.suffix.lower() == ".txt" and not path.name.startswith(".")


def load_corpus(data_dir: str) -> tuple[list[dict[str, Any]], dict[int, str], dict[str, int]]:
    """Load non-empty text documents grouped by author directory.

    Args:
        data_dir: Path to the root directory containing one subdirectory per author.

    Returns:
        A tuple containing:
            - A list of document dictionaries. Each dictionary contains the author name,
              numeric label, file path, document ID, and text content.
            - A mapping from numeric label IDs to author names.
            - A mapping from author names to numeric label IDs.
    """
    root = Path(data_dir)
    if not root.exists():
        raise FileNotFoundError(f"No existe el directorio: {data_dir}")

    author_dirs = [p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")]
    author_dirs = sorted(author_dirs, key=lambda x: x.name)
    authors = [p.name for p in author_dirs]
    label2id = {author: i for i, author in enumerate(authors)}
    id2label = {i: author for author, i in label2id.items()}

    documents: list[dict[str, Any]] = []
    for author_dir in author_dirs:
        author = author_dir.name
        txt_files = sorted([p for p in author_dir.iterdir() if is_valid_text_file(p)])
        for txt_path in txt_files:
            text = txt_path.read_text().strip()
            if not text:
                continue
            documents.append(
                {
                    "author": author,
                    "label": label2id[author],
                    "path": str(txt_path),
                    "doc_id": txt_path.stem,
                    "text": text,
                }
            )
    return documents, id2label, label2id


def split_documents_by_author(
    documents: list[dict[str, Any]],
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
    seed: int | float | str | bytes | bytearray | None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], list[dict[str, Any]]]:
    """Split documents into train, validation, and test sets per author.

    Args:
        documents: Documents to split. Each document is expected to contain an
            `"author"` key.
        train_ratio: Proportion of each author's documents assigned to the
            training split.
        val_ratio: Proportion of each author's documents assigned to the
            validation split.
        test_ratio: Proportion of each author's documents assigned to the test
            split.
        seed: Seed used to initialize the random number generator for
            reproducible shuffling.

    Returns:
        A tuple containing the training, validation, and test document lists.
    """
    total = train_ratio + val_ratio + test_ratio
    if not math.isclose(total, 1.0, rel_tol=1e-6):
        raise ValueError("train_ratio + val_ratio + test_ratio debe sumar 1.0")

    rng = random.Random(seed)
    by_author: defaultdict[str, list[dict[str, Any]]] = defaultdict(list)
    for doc in documents:
        by_author[doc["author"]].append(doc)

    train_docs: list[dict[str, Any]] = []
    val_docs: list[dict[str, Any]] = []
    test_docs: list[dict[str, Any]] = []

    for author, docs in by_author.items():
        rng.shuffle(docs)
        n = len(docs)
        if n < 3:
            raise ValueError(f"El autor '{author}' tiene solo {n} obras.")

        n_train = max(1, int(round(n * train_ratio)))
        n_val = max(1, int(round(n * val_ratio)))
        n_test = n - n_train - n_val

        if n_test < 1:
            n_test = 1
            if n_train > n_val:
                n_train -= 1
            else:
                n_val -= 1

        while n_train + n_val + n_test > n:
            if n_train > 1:
                n_train -= 1
            elif n_val > 1:
                n_val -= 1
            else:
                n_test -= 1

        while n_train + n_val + n_test < n:
            n_train += 1

        train_docs.extend(docs[:n_train])
        val_docs.extend(docs[n_train : n_train + n_val])
        test_docs.extend(docs[n_train + n_val :])

    return train_docs, val_docs, test_docs


def print_split_summary(
    train_docs: list[dict[str, Any]],
    val_docs: list[dict[str, Any]],
    test_docs: list[dict[str, Any]],
):
    """Print the number of documents per split and per author.

    Args:
        train_docs: Documents assigned to the training split. Each document is
            expected to contain an `"author"` key.
        val_docs: Documents assigned to the validation split. Each document is
            expected to contain an `"author"` key.
        test_docs: Documents assigned to the test split. Each document is
            expected to contain an `"author"` key.
    """

    def summarize(name: str, docs: list[dict[str, Any]]):
        """Print the number of documents for one split, grouped by author.

        Args:
            name: Name of the split to display.
            docs: Documents in the split. Each document is expected to contain
                an `"author"` key.
        """
        cnt = Counter(doc["author"] for doc in docs)
        print(f"\n[{name}] {len(docs)} documentos")
        for author, n in sorted(cnt.items()):
            print(f"  - {author}: {n}")

    summarize("TRAIN", train_docs)
    summarize("VAL", val_docs)
    summarize("TEST", test_docs)


def docs_to_df(docs: Sequence[Mapping[str, Any]], split_name: str) -> pd.DataFrame:
    """Convert document metadata into a split-labeled DataFrame.

    Builds a DataFrame with one row per document. Each row contains the split
    name and selected metadata fields from the corresponding document mapping.

    Args:
        docs: Sequence of document metadata mappings. Each document is expected
            to contain `author`, `doc_id`, and `path` keys.
        split_name: Name of the dataset split assigned to every document in
            `docs`.

    Returns:
        A DataFrame with `split`, `author`, `doc_id`, and `path` columns.
    """
    return pd.DataFrame(
        [
            {
                "split": split_name,
                "author": doc["author"],
                "doc_id": doc["doc_id"],
                "path": doc["path"],
            }
            for doc in docs
        ]
    )


def chunk_document(
    text: str,
    tokenizer: Callable[..., dict[str, list[list[int]]]],
    max_length: int,
    stride: int,
) -> list[dict[str, list[int]]]:
    """Split a document into tokenized chunks.

    Args:
        text: Document text to tokenize and split.
        tokenizer: Tokenizer callable compatible with the Hugging Face tokenizer
            API. It must return `"input_ids"` and `"attention_mask"` entries.
        max_length: Maximum number of tokens per chunk, including special tokens.
        stride: Number of overlapping tokens between consecutive chunks when
            truncation produces overflowing tokens.

    Returns:
        A list of chunk dictionaries. Each dictionary contains:
            - `"input_ids"`: Token IDs for one chunk.
            - `"attention_mask"`: Attention mask for one chunk.
    """
    enc = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=False,
    )

    chunks: list[dict[str, list[int]]] = []
    input_ids_list = enc["input_ids"]
    attention_mask_list = enc["attention_mask"]

    for input_ids, attention_mask in zip(input_ids_list, attention_mask_list):
        chunks.append(
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }
        )

    return chunks


def build_chunk_examples(
    docs: list[dict[str, Any]],
    tokenizer: Callable[..., dict[str, list[list[int]]]],
    max_length: int,
    stride: int,
) -> list[dict[str, Any]]:
    """Build chunk-level examples from document-level records.

    Args:
        docs: Document records to convert into chunk examples. Each document is
            expected to contain `"text"`, `"label"`, `"doc_id"`, `"author"`, and
            `"path"` keys.
        tokenizer: Tokenizer callable passed to `chunk_document`.
        max_length: Maximum number of tokens per chunk, including special tokens.
        stride: Number of overlapping tokens between consecutive chunks when
            truncation produces overflowing tokens.

    Returns:
        A list of chunk-level example dictionaries. Each example contains token
        IDs, an attention mask, the document label, document metadata, and a
        zero-based `chunk_id`.
    """
    examples: list[dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_document(doc["text"], tokenizer, max_length, stride)
        for idx, chunk in enumerate(chunks):
            examples.append(
                {
                    "input_ids": chunk["input_ids"],
                    "attention_mask": chunk["attention_mask"],
                    "labels": doc["label"],
                    "doc_id": doc["doc_id"],
                    "author": doc["author"],
                    "path": doc["path"],
                    "chunk_id": idx,
                }
            )
    return examples


class AuthorshipChunkDataset(Dataset[dict[str, Any]]):
    """PyTorch dataset for authorship chunk examples.

    Args:
        examples: Chunk-level examples. Each example is expected to contain
            `"input_ids"`, `"attention_mask"`, and `"labels"` keys.

    Attributes:
        examples: Stored chunk-level examples used by the dataset.

    Notes:
        The input examples are stored by reference, not copied. Mutating the
        original list or its dictionaries after initialization affects this
        dataset.
    """

    def __init__(self, examples: list[dict[str, Any]]) -> None:
        self.examples = examples

    def __len__(self) -> int:
        """Return the number of examples in the dataset.

        Returns:
            Number of stored examples.
        """
        return len(self.examples)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        """Return model inputs for one example.

        Args:
            idx: Zero-based index of the example to retrieve.

        Returns:
            A dictionary containing `"input_ids"`, `"attention_mask"`, and
            `"labels"` for the selected example.
        """
        ex = self.examples[idx]
        return {
            "input_ids": ex["input_ids"],
            "attention_mask": ex["attention_mask"],
            "labels": ex["labels"],
        }


def compute_metrics(eval_pred: tuple[Any, Any]) -> dict[str, float]:
    """Compute classification metrics from model predictions.

    Args:
        eval_pred: Tuple containing logits and true labels, as commonly passed
            by Hugging Face `Trainer`. The first item is expected to be an
            array-like object of logits, and the second item is expected to be
            an array-like object of label IDs.

    Returns:
        A dictionary containing accuracy, macro-averaged F1, and
        weighted-averaged F1 scores.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }


@dataclass
class DocumentLevelResult:
    y_true: list
    y_pred: list
    probs: dict


def aggregate_document_predictions(
    trainer: Any,
    examples: list[dict[str, Any]],
) -> DocumentLevelResult:
    """Aggregate chunk-level predictions into document-level predictions.

    Args:
        trainer: Trainer-like object with a `predict()` method. The method is
            expected to accept an `AuthorshipChunkDataset` and return an object
            with a `predictions` attribute containing chunk-level logits.
        examples: Chunk-level examples. Each example is expected to contain
            `"doc_id"` and `"labels"` keys.

    Returns:
        A document-level result containing true labels, predicted labels, and
        averaged class probabilities for each document ID.
    """
    dataset = AuthorshipChunkDataset(examples)
    pred_output = trainer.predict(dataset)
    logits = pred_output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    by_doc_probs: defaultdict[str, list[np.ndarray]] = defaultdict(list)
    by_doc_true: dict[str, Any] = {}
    for ex, pr in zip(examples, probs):
        by_doc_probs[ex["doc_id"]].append(pr)
        by_doc_true[ex["doc_id"]] = ex["labels"]

    y_true: list[int] = []
    y_pred: list[int] = []
    probs_out: dict[str, list[float]] = {}

    for doc_id, prob_list in by_doc_probs.items():
        avg_prob = np.mean(prob_list, axis=0)
        pred = int(np.argmax(avg_prob))
        true = int(by_doc_true[doc_id])
        y_true.append(true)
        y_pred.append(pred)
        probs_out[doc_id] = avg_prob.tolist()

    return DocumentLevelResult(y_true=y_true, y_pred=y_pred, probs=probs_out)


def evaluate_and_print(
    y_true: Sequence[int],
    y_pred: Sequence[int],
    id2label: dict[int, str],
    title: str,
) -> dict[str, Any]:
    """Evaluate classification predictions and print a summary report.

    Args:
        y_true: True class label IDs.
        y_pred: Predicted class label IDs.
        id2label: Mapping from class label IDs to display names.
        title: Title printed above the evaluation results.

    Returns:
        A dictionary containing accuracy, macro-F1, weighted-F1, and the
        confusion matrix as a nested list.
    """
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    labels_sorted = [id2label[i] for i in sorted(id2label.keys())]

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"Accuracy    : {acc:.4f}")
    print(f"Macro-F1    : {macro_f1:.4f}")
    print(f"Weighted-F1 : {weighted_f1:.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=labels_sorted,
            digits=4,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=list(sorted(id2label.keys())))
    print("Confusion matrix:")
    print(cm)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "confusion_matrix": cm.tolist(),
    }

## Carga de datos y split del dataset

In [6]:
documents, id2label, label2id = load_corpus(DATA_DIR)
print(f"Documentos cargados: {len(documents)}")
print("Autores:", list(label2id.keys()))

train_docs, val_docs, test_docs = split_documents_by_author(documents, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, SEED)
print_split_summary(train_docs, val_docs, test_docs)

Documentos cargados: 95
Autores: ['anna_katharine_green', 'arthur_conan_doyle', 'arthur_morrison', 'gilbert_keith_chesterton', 'richard_austin_freeman', 'wilkie_collins']

[TRAIN] 66 documentos
  - anna_katharine_green: 24
  - arthur_conan_doyle: 8
  - arthur_morrison: 4
  - gilbert_keith_chesterton: 13
  - richard_austin_freeman: 10
  - wilkie_collins: 7

[VAL] 15 documentos
  - anna_katharine_green: 5
  - arthur_conan_doyle: 2
  - arthur_morrison: 1
  - gilbert_keith_chesterton: 3
  - richard_austin_freeman: 2
  - wilkie_collins: 2

[TEST] 14 documentos
  - anna_katharine_green: 5
  - arthur_conan_doyle: 2
  - arthur_morrison: 1
  - gilbert_keith_chesterton: 2
  - richard_austin_freeman: 3
  - wilkie_collins: 1


In [7]:
split_df = pd.concat(
    [
        docs_to_df(train_docs, "train"),
        docs_to_df(val_docs, "val"),
        docs_to_df(test_docs, "test"),
    ],
    ignore_index=True,
)

# Mostramos las obras que se han asignado al conjunto de test
test_df = split_df[split_df["split"] == "test"].sort_values(["author", "doc_id"])
test_df

,split,author,doc_id,path
84,test,anna_katharine_green,akg-a_difficult_problem,/content/drive/MyDrive/corpus/anna_katharine_g...
85,test,anna_katharine_green,akg-dark_hollow,/content/drive/MyDrive/corpus/anna_katharine_g...
83,test,anna_katharine_green,akg-the_circular_study,/content/drive/MyDrive/corpus/anna_katharine_g...
81,test,anna_katharine_green,akg-the_woman_in_the_alcove,/content/drive/MyDrive/corpus/anna_katharine_g...
82,test,anna_katharine_green,akg-x_y_z_a_detective_story,/content/drive/MyDrive/corpus/anna_katharine_g...
86,test,arthur_conan_doyle,acd-the_hound_of_the_baskervilles,/content/drive/MyDrive/corpus/arthur_conan_doy...
87,test,arthur_conan_doyle,acd-the_lost_world,/content/drive/MyDrive/corpus/arthur_conan_doy...
88,test,arthur_morrison,am-the_hole_in_the_wall,/content/drive/MyDrive/corpus/arthur_morrison/...
89,test,gilbert_keith_chesterton,gkc-the_donnington_affair,/content/drive/MyDrive/corpus/gilbert_keith_ch...
90,test,gilbert_keith_chesterton,gkc-the_wisdom_of_father_brown,/content/drive/MyDrive/corpus/gilbert_keith_ch...


In [8]:
split_csv_path = os.path.join(OUTPUT_DIR, "document_splits.csv")
split_df.to_csv(split_csv_path, index=False, encoding="utf-8")

print("Split guardado en:", split_csv_path)

Split guardado en: /content/drive/MyDrive/outputs/authorship_transformer/document_splits.csv


## Tokenización y chunking de los documentos

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

train_examples = build_chunk_examples(train_docs, tokenizer, MAX_LENGTH, STRIDE)
val_examples = build_chunk_examples(val_docs, tokenizer, MAX_LENGTH, STRIDE)
test_examples = build_chunk_examples(test_docs, tokenizer, MAX_LENGTH, STRIDE)

print("Chunks TRAIN:", len(train_examples))
print("Chunks VAL  :", len(val_examples))
print("Chunks TEST :", len(test_examples))

train_dataset = AuthorshipChunkDataset(train_examples)
val_dataset = AuthorshipChunkDataset(val_examples)
test_dataset = AuthorshipChunkDataset(test_examples)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Chunks TRAIN: 38097
Chunks VAL  : 9875
Chunks TEST : 6519


## Definición del modelo y entrenamiento

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if FP16 else None)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Los pesos marcados como `UNEXPECTED` pertenecen a la cabeza de lenguaje de RoBERTa, normalmente usada para tareas tipo `masked language modeling`. Como nosotros estmos creando un modelo de clasificación, esa cabeza no se necesita y se ignorará.

Los pesos marcados como `MISSING` significan que la cabeza de clasificación no existía en roberta-base, así que Hugging Face la crea desde cero con pesos aleatorios.

In [11]:
print("=== Diagnóstico GPU ===")
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA compilado en PyTorch:", torch.backends.cuda.is_built())
print("torch.version.cuda:", torch.version.cuda)

if torch.cuda.is_available():
    print("Número de GPUs:", torch.cuda.device_count())
    print("GPU activa:", torch.cuda.current_device())
    print("Nombre GPU:", torch.cuda.get_device_name(0))
    print("Modelo en:", next(model.parameters()).device)
else:
    print("No se detecta GPU; se está usando CPU.")

=== Diagnóstico GPU ===
CUDA disponible: True
CUDA compilado en PyTorch: True
torch.version.cuda: 12.8
Número de GPUs: 1
GPU activa: 0
Nombre GPU: Tesla T4
Modelo en: cpu


### Argumentos de entrenamiento

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    seed=SEED,
    fp16=FP16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    report_to="none",
)

### Entrenamiento del modelo

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.109026,0.620295,0.893671,0.862762,0.900422
2,0.038412,0.602379,0.920709,0.890356,0.923398
3,0.012754,0.427599,0.946329,0.929678,0.946460
4,0.000484,0.640570,0.924152,0.906654,0.927150
5,0.004507,0.555068,0.934886,0.914033,0.934678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=23815, training_loss=0.09319147412113199, metrics={'train_runtime': 3442.831, 'train_samples_per_second': 110.656, 'train_steps_per_second': 13.835, 'total_flos': 2.506025468146176e+16, 'train_loss': 0.09319147412113199, 'epoch': 5.0})

### Guardado del modelo entrenado

In [14]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Modelo guardado en", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en /content/drive/MyDrive/outputs/authorship_transformer


## Evaluación del modelo

### Evaluación a nivel de chunk

In [15]:
# Evaluación por chunk
chunk_pred = trainer.predict(test_dataset)
chunk_logits = chunk_pred.predictions
chunk_labels = chunk_pred.label_ids
chunk_preds = np.argmax(chunk_logits, axis=-1)

chunk_metrics = evaluate_and_print(
    y_true=chunk_labels.tolist(),
    y_pred=chunk_preds.tolist(),
    id2label=id2label,
    title="RESULTADOS TEST - NIVEL CHUNK",
)


RESULTADOS TEST - NIVEL CHUNK
Accuracy    : 0.8645
Macro-F1    : 0.8140
Weighted-F1 : 0.8638

Classification report:
                          precision    recall  f1-score   support

    anna_katharine_green     0.9478    0.9401    0.9440      1604
      arthur_conan_doyle     0.8745    0.9638    0.9170       911
         arthur_morrison     0.4815    0.4435    0.4617       469
gilbert_keith_chesterton     0.6860    0.9486    0.7962       525
  richard_austin_freeman     0.9402    0.7825    0.8542      1669
          wilkie_collins     0.8991    0.9232    0.9110      1341

                accuracy                         0.8645      6519
               macro avg     0.8048    0.8336    0.8140      6519
            weighted avg     0.8710    0.8645    0.8638      6519

Confusion matrix:
[[1508   50    5    6   23   12]
 [   5  878    0    5   15    8]
 [  14   35  208   95   38   79]
 [   3   10    4  498    3    7]
 [  48   28  137  117 1306   33]
 [  13    3   78    5    4 1238]]


### Evaluación a nivel de documento

In [16]:
# Evaluación por documento
doc_result = aggregate_document_predictions(trainer, test_examples)

doc_metrics = evaluate_and_print(
    y_true=doc_result.y_true,
    y_pred=doc_result.y_pred,
    id2label=id2label,
    title="RESULTADOS TEST - NIVEL DOCUMENTO",
)


RESULTADOS TEST - NIVEL DOCUMENTO
Accuracy    : 1.0000
Macro-F1    : 1.0000
Weighted-F1 : 1.0000

Classification report:
                          precision    recall  f1-score   support

    anna_katharine_green     1.0000    1.0000    1.0000         5
      arthur_conan_doyle     1.0000    1.0000    1.0000         2
         arthur_morrison     1.0000    1.0000    1.0000         1
gilbert_keith_chesterton     1.0000    1.0000    1.0000         2
  richard_austin_freeman     1.0000    1.0000    1.0000         3
          wilkie_collins     1.0000    1.0000    1.0000         1

                accuracy                         1.0000        14
               macro avg     1.0000    1.0000    1.0000        14
            weighted avg     1.0000    1.0000    1.0000        14

Confusion matrix:
[[5 0 0 0 0 0]
 [0 2 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 2 0 0]
 [0 0 0 0 3 0]
 [0 0 0 0 0 1]]


## Guardado de resultados y métricas

In [17]:
metadata = {
    "config": {
        "data_dir": DATA_DIR,
        "model_name": MODEL_NAME,
        "output_dir": OUTPUT_DIR,
        "max_length": MAX_LENGTH,
        "stride": STRIDE,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "epochs": EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
    },
    "authors": list(label2id.keys()),
    "label2id": label2id,
    "id2label": id2label,
    "n_documents_total": len(documents),
    "n_train_docs": len(train_docs),
    "n_val_docs": len(val_docs),
    "n_test_docs": len(test_docs),
    "n_train_chunks": len(train_examples),
    "n_val_chunks": len(val_examples),
    "n_test_chunks": len(test_examples),
    "document_splits": {
        "train": [{"author": d["author"], "doc_id": d["doc_id"], "path": d["path"]} for d in train_docs],
        "val": [{"author": d["author"], "doc_id": d["doc_id"], "path": d["path"]} for d in val_docs],
        "test": [{"author": d["author"], "doc_id": d["doc_id"], "path": d["path"]} for d in test_docs],
    },
    "chunk_metrics": chunk_metrics,
    "document_metrics": doc_metrics,
    "test_doc_probabilities": doc_result.probs,
}

results_path = os.path.join(OUTPUT_DIR, "results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Resultados guardados en", results_path)

Resultados guardados en /content/drive/MyDrive/outputs/authorship_transformer/results.json


## Cargar el modelo entrenado

Cargamos el modelo entrenado para ver que se ha guardado correctamente y que se puede usar para hacer predicciones.

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [19]:
model.config

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "anna_katharine_green",
    "1": "arthur_conan_doyle",
    "2": "arthur_morrison",
    "3": "gilbert_keith_chesterton",
    "4": "richard_austin_freeman",
    "5": "wilkie_collins"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "anna_katharine_green": 0,
    "arthur_conan_doyle": 1,
    "arthur_morrison": 2,
    "gilbert_keith_chesterton": 3,
    "richard_austin_freeman": 4,
    "wilkie_collins": 5
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "problem_type": "sin

In [20]:
summary(model)

Layer (type:depth-idx)                                       Param #
RobertaForSequenceClassification                             --
├─RobertaModel: 1-1                                          --
│    └─RobertaEmbeddings: 2-1                                --
│    │    └─Embedding: 3-1                                   38,603,520
│    │    └─Embedding: 3-2                                   768
│    │    └─LayerNorm: 3-3                                   1,536
│    │    └─Dropout: 3-4                                     --
│    │    └─Embedding: 3-5                                   394,752
│    └─RobertaEncoder: 2-2                                   --
│    │    └─ModuleList: 3-6                                  85,054,464
├─RobertaClassificationHead: 1-2                             --
│    └─Linear: 2-3                                           590,592
│    └─Dropout: 2-4                                          --
│    └─Linear: 2-5                                           4,614
To